# StreamSense — Live Serving + Dashboard (Colab)

Brings up the **real serving path** natively on Colab's Linux (no Docker, no emulation) and launches
the **StreamSense Explorer** dashboard, which answers every request through the serving APIs.

Serves three models:
- **Retrieval** (two-tower) → `tensorflow_model_server` on **:8501**
- **TFX ranker** (Keras SavedModel) → `tensorflow_model_server` on **:8502**  *(needs `models/ranking_tf` — see note)*
- **PyTorch ranker** → **TorchServe** on **:8080**

Dashboard modes: **Testing** = single TFX ranker (:8502); **Prod** = retrieval (:8501) → PyTorch ranker (:8080).

> **Run cells in order.** Prod mode works from the committed models. Testing mode needs the TFX
> SavedModel committed to `models/ranking_tf` (one-time: in your TFX-pipeline Colab run
> `!cp -r artifacts/ranking_tf models/ranking_tf` and commit it).

## 1. Clone + install serving stack (native, no Docker)

In [ ]:
!git clone https://github.com/techykajal/streamsense-ott-recsys.git 2>/dev/null || true
%cd streamsense-ott-recsys

# TensorFlow Serving binary
!echo "deb http://storage.googleapis.com/tensorflow-serving-apt stable tensorflow-model-server tensorflow-model-server-universal" > /etc/apt/sources.list.d/tensorflow-serving.list
!curl -s https://storage.googleapis.com/tensorflow-serving-apt/tensorflow-serving.release.pub.gpg | apt-key add - 2>/dev/null
!apt-get update -q && apt-get install -y -q tensorflow-model-server

# TorchServe (+ Java) and the dashboard stack
!apt-get install -y -q openjdk-17-jdk-headless
!pip -q install torchserve torch-model-archiver streamlit tensorflow-recommenders pyarrow
print("install done")

## 2. Build the dashboard's data (parquet) + confirm models

In [ ]:
!python src/features.py
import os
print("retrieval model :", "OK" if os.path.exists("models/retrieval") else "MISSING")
print("torch ranker    :", "OK" if os.path.exists("models/ranker.pt") else "MISSING (run src/ranking_torch.py)")
print("TFX ranker      :", "OK" if os.path.exists("models/ranking_tf") else "not committed -> Testing mode disabled")
# make sure the torch ranker exists (train quickly if the committed one is absent)
if not os.path.exists("models/ranker.pt"):
    !python src/ranking_torch.py && mkdir -p models && cp artifacts/ranker.pt artifacts/id_maps.json models/

## 3. Start TensorFlow Serving (retrieval :8501, TFX ranker :8502)

In [ ]:
import subprocess, os, time
ROOT = os.getcwd()
# retrieval needs a versioned dir
os.makedirs("serve/retrieval/1", exist_ok=True)
!cp -r models/retrieval/* serve/retrieval/1/ 2>/dev/null || true

subprocess.Popen(["tensorflow_model_server","--rest_api_port=8501",
                  "--model_name=retrieval", f"--model_base_path={ROOT}/serve/retrieval"])
if os.path.exists("models/ranking_tf"):
    subprocess.Popen(["tensorflow_model_server","--rest_api_port=8502",
                      "--model_name=ranker", f"--model_base_path={ROOT}/models/ranking_tf"])
    print("TFX ranker serving on :8502")
else:
    print("models/ranking_tf missing -> :8502 not started (Testing mode will show ranker down)")
time.sleep(8)
!curl -s http://localhost:8501/v1/models/retrieval | head -c 200

## 4. Start TorchServe for the PyTorch ranker (:8080)

In [ ]:
import os
os.makedirs("model_store", exist_ok=True)
# handler imports ranking_torch + loads ranker.pt from the model dir
!torch-model-archiver --model-name ranker --version 1.0     --serialized-file models/ranker.pt     --handler serving/torchserve_handler.py     --extra-files "src/ranking_torch.py,models/ranker.pt"     --export-path model_store -f
!torchserve --stop 2>/dev/null; sleep 2
!torchserve --start --ncs --model-store model_store --models ranker=ranker.mar --disable-token-auth 2>/dev/null
import time; time.sleep(12)
!curl -s http://localhost:8080/ping

## 5. Smoke-test the serving APIs

In [ ]:
import sys; sys.path.append("serving")
# TorchServe: 2-row request
import requests, json
rows=[{"user":0,"movie":1,"seg":0},{"user":0,"movie":2,"seg":0}]
try:
    print("torch:", requests.post("http://localhost:8080/predictions/ranker", json=rows, timeout=20).text[:200])
except Exception as e: print("torch err:", e)
# retrieval
print("retrieval:", requests.post("http://localhost:8501/v1/models/retrieval:predict",
      json={"inputs":{"user_id":["1"],"segment":[0]}}, timeout=20).text[:200])

## 6. Launch the dashboard + public URL

In [ ]:
import subprocess, time
subprocess.Popen(["streamlit","run","app/streamsense_explorer.py",
                  "--server.port","8500","--server.headless","true"])
time.sleep(8)
print("If localtunnel asks for a password, it is this IP:")
!curl -s https://loca.lt/mytunnelpassword || curl -s ifconfig.me
print("\nOpen the https URL printed below:")
!npx --yes localtunnel --port 8500

## 7. Using the demo

In the dashboard sidebar, switch **Testing** (single TFX ranker) vs **Prod** (retrieval → PyTorch
ranker). Tab 1: pick a user → live recommendations. Tab 2: pick a movie → similar titles + likely
viewers. Tab 3: metrics (run `notebooks/StreamSense_Evaluation.ipynb` to populate).

**Stop everything:** `!torchserve --stop` and restart the runtime.

*First-run note:* TorchServe + the model-server binary occasionally need a few seconds more to warm
up — if a smoke test is empty, re-run that cell before launching the dashboard.